# Notebook 02 — 1D Linear Interpolation & Image Nearest-Neighbor Resizing

**Session:** Interpolation Practice Lab  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement **1D Linear Interpolation** (`linear_interpolation`, `li_1d`) for Problem 7 and apply **Nearest-Neighbor Image Resizing** (`resize_nni`) to `../data/tree.jpg` ($400 \times 400 \to 1200 \times 1200$) for Problem 8.


In [ ]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

IMAGE_PATH = Path("../data/tree.jpg")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Problem 7 — 1D Linear Interpolation (`linear_interpolation` & `li_1d`)

Given two points $p_1 = (x_1, y_1)$ and $p_2 = (x_2, y_2)$, the linear interpolant at query $x$ is:

$$y = \\frac{x_2 - x}{x_2 - x_1} y_1 + \\frac{x - x_1}{x_2 - x_1} y_2$$

To upsample `data` of length $n$ to length $n \cdot k$, each output index $i \in [0, nk - k]$ maps to fractional source coordinate $x = i / k$ bounded by $x_1 = \lfloor x \rfloor$ and $x_2 = \min(\lceil x \rceil, n - 1)$.


In [ ]:
def linear_interpolation(
    p1: tuple[float, float],
    p2: tuple[float, float],
    x: float,
) -> float:
    """Linearly interpolate y at query coordinate x between p1=(x1, y1) and p2=(x2, y2)."""
    x1, y1 = p1
    x2, y2 = p2
    if x1 != x2:
        return ((x2 - x) / (x2 - x1)) * y1 + ((x - x1) / (x2 - x1)) * y2
    return float(y1)


def li_1d(data: list[float], k: int, clamp_tail: bool = False) -> list[float]:
    """Upsample a 1D sequence by factor k using linear interpolation.

    When clamp_tail=False, the trailing (k - 1) extrapolated entries are left as 0.0
    (matching the slide exercise specification). When clamp_tail=True, they clamp
    to data[-1].
    """
    if k <= 0:
        raise ValueError("Scale factor k must be positive.")
    n = len(data)
    result = [float(data[-1]) if clamp_tail else 0.0] * (n * k)
    for i in range(n * k - (k - 1)):
        x = i / k
        x1 = math.floor(x)
        x2 = min(math.ceil(x), n - 1)
        result[i] = linear_interpolation((x1, data[x1]), (x2, data[x2]), x)
    return result


# Slide numerical checks
assert math.isclose(linear_interpolation((1, 1), (3, 3), 2.0), 2.0)
assert math.isclose(linear_interpolation((1, 1), (3, 3), 1.1), 1.1)
assert math.isclose(linear_interpolation((0, 0), (4, 2), 1.0), 0.5)

# Problem 7 test cases (ignoring trailing k - 1 positions)
assert li_1d([1, 2, 3], 2)[:-1] == [1.0, 1.5, 2.0, 2.5, 3.0]
assert li_1d([1, 4, 7], 3)[:-2] == [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
print("Problem 7 passed:", li_1d([1, 4, 7], 3, clamp_tail=True))

In [ ]:
# Visualize 1D Nearest-Neighbor vs. Linear Interpolation
sample = [1.0, 4.0, 2.0, 6.0]
k = 4
nni_up = [sample[i // k] for i in range(len(sample) * k)]
lin_up = li_1d(sample, k, clamp_tail=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(len(sample)) * k, sample, "ro", markersize=8, label="Observed samples")
ax.step(range(len(nni_up)), nni_up, where="post", linestyle="--", label="1D Nearest-Neighbor")
ax.plot(range(len(lin_up)), lin_up, "-o", markersize=4, label="1D Linear Interpolation")
ax.set_title("1D Upsampling (k = 4): Nearest-Neighbor vs. Linear")
ax.set_xlabel("Output Index")
ax.set_ylabel("Value")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Problem 8 (Part A) — Image Resizing via Nearest-Neighbor (`resize_nni`)

Load `../data/tree.jpg` in grayscale (`(400, 400)`), convert it to a Python 2D list, resize it $3\times$ to `(1200, 1200)`, and verify exact parity with `cv2.resize(..., interpolation=cv2.INTER_NEAREST)`.


In [ ]:
def resize_nni(
    source: list[list[int]],
    source_h: int,
    source_w: int,
    target_h: int,
    target_w: int,
) -> list[list[int]]:
    """Resize a 2D grayscale list using Nearest-Neighbor Interpolation."""
    new_data = [[0] * target_w for _ in range(target_h)]
    w_scale_factor = source_w / target_w
    h_scale_factor = source_h / target_h
    for i in range(target_h):
        src_i = min(source_h - 1, math.floor(i * h_scale_factor))
        for j in range(target_w):
            src_j = min(source_w - 1, math.floor(j * w_scale_factor))
            new_data[i][j] = source[src_i][src_j]
    return new_data


img_gray = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
assert img_gray is not None, f"Could not load {IMAGE_PATH}"
height, width = img_gray.shape

resized_list = resize_nni(img_gray.tolist(), height, width, height * 3, width * 3)
resized_nni = np.array(resized_list, dtype=np.uint8)
cv_nni = cv2.resize(img_gray, (width * 3, height * 3), interpolation=cv2.INTER_NEAREST)

assert resized_nni.shape == (1200, 1200)
assert np.array_equal(resized_nni, cv_nni)
cv2.imwrite(str(OUTPUT_DIR / "tree_nni_3x.jpg"), resized_nni)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_gray, cmap="gray")
axes[0].set_title(f"Original {img_gray.shape}")
axes[0].axis("off")

axes[1].imshow(resized_nni, cmap="gray")
axes[1].set_title(f"Nearest-Neighbor 3x {resized_nni.shape}")
axes[1].axis("off")

plt.tight_layout()
plt.show()
print("Exact match with cv2.INTER_NEAREST:", np.array_equal(resized_nni, cv_nni))